
# 01b_Metadaten_Cleaning_QC

Dieses Notebook führt eine **strukturierte Qualitätskontrolle und Reinigung** der in **01a** erzeugten roh-harmonisierten Metadaten durch und erzeugt eine **valide Basistabelle** als Grundlage für das Feature-Engineering (02a).

**Eingabe (aus 01a):**
- `metadata_raw.(parquet|csv)` – roh harmonisierte Tabelle (1 Zeile pro `video_id`)
- `metadata_provenance.csv` – Quelle & Gruppe je `video_id`

**Ausgabe (neben diesem Notebook):**
- `metadata_base.(parquet|csv)` – **validierte** Basistabelle
- `schema_metadata_base.json` – Schema-Vertrag für Downstream-Schritte
- `leakage_scan.csv` – Liste potenziell leaky Spaltennamen
- `missing_video_ids.csv` – nur wenn `data/labels.csv` existiert

> **Wichtig:** Dieses Notebook **erzeugt noch keine Features**, sondern schafft einen **verlässlichen Zustand** der Metadaten.



## Governance & Guardrails

- **Keine Leakage**: Spalten, die spätere Outcomes beschreiben (z. B. `likes`, `views`, `comments`, `shares`), werden **nicht** als Features zugelassen; hier werden sie nur dokumentiert/gescannt.
- **Zeit in UTC**: Zeitspalten werden auf **UTC** normalisiert/geprüft.
- **Schema-Vertrag**: Wir definieren ein **Minimal-Schema** (Pflichtspalten + erwartete Typen) und speichern es.
- **Determinismus**: Alle Entscheidungen sind deterministisch (keine Zufallsoperationen).
- **Provenienz**: Beibehaltung der Herkunfts-Infos (für Bericht & Debugging).


## 1) Setup & Inputs

In [ ]:

from pathlib import Path
import pandas as pd, numpy as np, json, re, warnings
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path("..").resolve()
NOTEBOOKS_DIR = Path(".").resolve()

RAW_PARQUET = NOTEBOOKS_DIR / "metadata_raw.parquet"
RAW_CSV = NOTEBOOKS_DIR / "metadata_raw.csv"
PROV_CSV = NOTEBOOKS_DIR / "metadata_provenance.csv"

print("Erwartete Inputs:")
print(" -", RAW_PARQUET, RAW_PARQUET.exists())
print(" -", RAW_CSV, RAW_CSV.exists())
print(" -", PROV_CSV, PROV_CSV.exists())

if RAW_PARQUET.exists():
    df = pd.read_parquet(RAW_PARQUET)
elif RAW_CSV.exists():
    df = pd.read_csv(RAW_CSV)
else:
    raise FileNotFoundError("Weder metadata_raw.parquet noch metadata_raw.csv gefunden. Bitte 01a ausführen.")

prov = None
if PROV_CSV.exists():
    prov = pd.read_csv(PROV_CSV)

print("Geladen:", df.shape)
df.head(3)



## 2) Schema-Entwurf (Minimalvertrag)

> 📌 **Platzhalter (Team-Entscheidung):**  
> - Müssen `title`/`description` **Pflicht** sein, oder reichen sie als **Optional**?  
> - Welche Felder sind **unbedingt** notwendig für eure Definition von Viralität?


In [ ]:

schema = {
    "required": ["video_id", "group"],
    "optional": [
        "title","description","uploader",
        "upload_time","create_time","timestamp","create_date",
        "duration_s",
        "creator_verified","creator_follower_count","creator_posts_count",
        "likes","comments","views","shares","engagement_score","rank",
        "source_file"
    ]
}

missing_required = [c for c in schema["required"] if c not in df.columns]
if missing_required:
    raise AssertionError(f"Pflichtspalten fehlen: {missing_required}")
print("Schema-Check (Presence) OK.")


## 3) Typen & Normalisierung

In [ ]:

def normalize_types(df):
    out = df.copy()
    if "video_id" in out.columns:
        out["video_id"] = out["video_id"].astype(str)
    if "creator_verified" in out.columns:
        out["creator_verified"] = out["creator_verified"].fillna(False).astype(bool)
    for c in ["duration_s","creator_follower_count","creator_posts_count",
              "likes","comments","views","shares","engagement_score","rank"]:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")
    for c in ["upload_time","create_time","timestamp","create_date"]:
        if c in out.columns:
            out[c] = pd.to_datetime(out[c], utc=True, errors="coerce")
    for c in ["title","description","uploader","source_file","group"]:
        if c in out.columns:
            out[c] = out[c].astype(str).str.strip()
    return out

df_n = normalize_types(df)
print("Normalisierung abgeschlossen.")
df_n.sample(3, random_state=0)


## 4) Deduplizierung & Schlüssel-Integrität

In [ ]:

df_n = df_n[df_n["video_id"].notna() & (df_n["video_id"] != "")]

time_col = next((c for c in ["upload_time","create_time","timestamp","create_date"] if c in df_n.columns), None)
if time_col:
    df_n = df_n.sort_values(["video_id", time_col], ascending=[True, False])

before = len(df_n)
df_n = df_n.drop_duplicates("video_id", keep="first").reset_index(drop=True)
after = len(df_n)
print(f"Dedupe: {before} → {after}")


## 5) Leakage-Scan

In [ ]:

import re
LEAKY_PAT = re.compile(r"(like|view|comment|share|play|impression|watch|engagement)(_)?(count|rate|s|7d|14d|total|score)?", re.I)
leaky_cols = [c for c in df_n.columns if LEAKY_PAT.search(c)]
leaky_cols


### Export des Leakage-Scans

In [ ]:

(pd.Series(leaky_cols, name="leaky_columns")
   .to_frame()
   .to_csv(NOTEBOOKS_DIR/"leakage_scan.csv", index=False))
print("Leakage-Report gespeichert:", (NOTEBOOKS_DIR/"leakage_scan.csv").resolve())


## 6) Labels mergen (optional)

In [ ]:

LABELS = PROJECT_ROOT / "data" / "labels.csv"
has_labels = LABELS.exists()
print("labels.csv vorhanden:", has_labels, "|", LABELS)

df_base = df_n.copy()

if has_labels:
    labels = pd.read_csv(LABELS)
    labels.columns = [re.sub(r"\W+","_", c.strip()).lower() for c in labels.columns]
    assert {"video_id","is_viral"}.issubset(labels.columns), "labels.csv muss Spalten video_id,is_viral enthalten."
    labels["video_id"] = labels["video_id"].astype(str)
    df_base = labels.merge(df_base, on="video_id", how="left", validate="one_to_one")
else:
    if "is_viral_proxy" not in df_base.columns and "group" in df_base.columns:
        df_base["is_viral_proxy"] = (df_base["group"].str.lower()=="top").astype(int)

print("Shape df_base:", df_base.shape)
df_base.head(3)


## 7) Missingness-Report & Coverage (optional)

In [ ]:

missing = df_base.isna().mean().sort_values(ascending=False)
print("Top fehlende Spalten:")
missing.head(20)


In [ ]:

if (PROJECT_ROOT / "data" / "labels.csv").exists():
    labels = pd.read_csv(PROJECT_ROOT / "data" / "labels.csv")
    labels.columns = [re.sub(r"\W+","_", c.strip()).lower() for c in labels.columns]
    labels_vids = set(labels["video_id"].astype(str))
    meta_vids = set(df_base["video_id"].astype(str))
    missing_ids = sorted(labels_vids - meta_vids)
    coverage = 1 - (len(missing_ids) / max(1, len(labels_vids)))
    print(f"Coverage: {coverage:.2%} | fehlend: {len(missing_ids)} von {len(labels_vids)}")
    import pandas as pd
    pd.DataFrame({"video_id": missing_ids}).to_csv(NOTEBOOKS_DIR/"missing_video_ids.csv", index=False)
    print("Liste fehlender IDs:", (NOTEBOOKS_DIR/"missing_video_ids.csv").resolve())
else:
    print("Kein Coverage-Check — es gibt keine labels.csv.")


## 8) Schema-Vertrag schreiben

In [ ]:

schema_contract = {
    "required": schema["required"],
    "optional": schema["optional"],
    "dtypes_snapshot": {c: str(dt) for c, dt in df_base.dtypes.items()}
}
out_schema = NOTEBOOKS_DIR / "schema_metadata_base.json"
out_schema.write_text(json.dumps(schema_contract, indent=2, ensure_ascii=False), encoding="utf-8")
print("Schema gespeichert:", out_schema.resolve())


## 9) Persistenz: metadata_base.(parquet|csv)

In [ ]:

def save_parquet_or_csv(df, base_name: str):
    base = Path(base_name)
    parquet_path = base.with_suffix(".parquet")
    csv_path = base.with_suffix(".csv")
    try:
        df.to_parquet(parquet_path, index=False)
        print("Gespeichert (Parquet):", parquet_path.resolve())
        return parquet_path
    except Exception as e:
        print("ℹ️ Parquet nicht verfügbar. Fallback → CSV.", "Grund:", repr(e))
        df.to_csv(csv_path, index=False)
        print("Gespeichert (CSV):", csv_path.resolve())
        return csv_path

out_written = save_parquet_or_csv(df_base, "metadata_base")
out_written



---

## ✅ Abschluss & Übergabe an 02a

Dieses Notebook hat:
- die roh-harmonisierten Metadaten **validiert & bereinigt**,  
- **Leakage-gefährliche Spalten** gelistet,  
- ein **Schema** gespeichert,  
- und `metadata_base.(parquet|csv)` als **verlässliche Grundlage** erzeugt.

**Nächster Schritt:** 📘 `02a_Metadaten_Features_Engineering.ipynb` – deterministisches Feature-Engineering.
